In [ ]:
# Se deben importar las librerías usadas durante el análisis.

from pathlib import Path
import pandas as pd
import plotly.express as px

In [ ]:
# Se deben definir y verificar la ruta del archivo de entregas y la carpeta de resultados.

SUPPLY_CHAIN_PATH = Path("../data/supply_chain.csv")
SUBMISSION_DIR = Path("../submission")
assert SUPPLY_CHAIN_PATH.exists(), f"No existe {SUPPLY_CHAIN_PATH}"
assert SUBMISSION_DIR.exists(), f"No existe {SUBMISSION_DIR}"

In [ ]:
# Se debe cargar el archivo y verificar sus tipos, tamaño y campos disponibles.

shipments = pd.read_csv(SUPPLY_CHAIN_PATH)
shipments.info()
shipments.head()

In [ ]:
# Se debe explorar un registro completo para reconocer el grano: una línea de envío.

shipments.loc[0]

In [ ]:
# Se debe verificar la unicidad de los identificadores, los faltantes y las categorías operativas.

quality_summary = pd.DataFrame(
    {
        "missing": shipments.isna().sum(),
        "distinct": shipments.nunique(dropna=True),
    }
)
assert shipments["ID"].is_unique, "Cada fila debe representar un envío distinto"
quality_summary.loc[
    [
        "ID",
        "Country",
        "Shipment Mode",
        "Scheduled Delivery Date",
        "Delivered to Client Date",
        "Line Item Value",
        "Freight Cost (USD)",
    ]
]

In [ ]:
# Se deben convertir las fechas y construir el KPI de días de diferencia frente a la fecha prometida.

for column in ["Scheduled Delivery Date", "Delivered to Client Date"]:
    shipments[column] = pd.to_datetime(
        shipments[column], format="%d-%b-%y", errors="coerce"
    )

assert (
    shipments[["Scheduled Delivery Date", "Delivered to Client Date"]]
    .notna()
    .all()
    .all()
)
shipments["DeliveryDaysVsSchedule"] = (
    shipments["Delivered to Client Date"] - shipments["Scheduled Delivery Date"]
).dt.days
shipments["IsLate"] = shipments["DeliveryDaysVsSchedule"] > 0
shipments["DeliveryMonth"] = (
    shipments["Delivered to Client Date"].dt.to_period("M").dt.to_timestamp()
)
shipments[
    [
        "Scheduled Delivery Date",
        "Delivered to Client Date",
        "DeliveryDaysVsSchedule",
        "IsLate",
    ]
].head()

In [ ]:
# Se debe limpiar el flete sin convertir los textos operativos en costos cero.

shipments["FreightCostUSD"] = pd.to_numeric(
    shipments["Freight Cost (USD)"], errors="coerce"
)
shipments["HasNumericFreightCost"] = shipments["FreightCostUSD"].notna()
freight_quality = (
    shipments.groupby("HasNumericFreightCost", dropna=False)
    .agg(
        shipments=("ID", "size"),
        line_item_value_usd=("Line Item Value", "sum"),
    )
    .reset_index()
)
freight_quality

In [ ]:
# ¿Cuál es el nivel general de cumplimiento y qué valor económico está expuesto a entregas tardías?

overall_kpis = pd.Series(
    {
        "envíos": shipments["ID"].size,
        "valor_total_usd": shipments["Line Item Value"].sum(),
        "cumplimiento_a_tiempo": 1 - shipments["IsLate"].mean(),
        "demora_mediana_días": shipments["DeliveryDaysVsSchedule"].median(),
        "valor_enviado_tarde_usd": shipments.loc[
            shipments["IsLate"], "Line Item Value"
        ].sum(),
    }
)

overall_kpis.to_csv(
    SUBMISSION_DIR / "overall_kpis.csv", index_label="indicador", header=["valor"]
)

overall_kpis

In [ ]:
# ¿Cómo se distribuyen los días de diferencia frente a la fecha prometida?

px.histogram(
    shipments,
    x="DeliveryDaysVsSchedule",
    nbins=45,
    title="Distribución de días frente a la fecha programada",
    labels={"DeliveryDaysVsSchedule": "Días: real menos programada", "count": "Envíos"},
    color_discrete_sequence=["#1f77b4"],
).add_vline(x=0, line_dash="dash", line_color="#333333")

In [ ]:
# ¿Qué modos de envío combinan alto volumen, incumplimiento y valor expuesto?

mode_summary = (
    shipments.groupby("Shipment Mode", dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        demora_promedio_días=("DeliveryDaysVsSchedule", "mean"),
        valor_total_usd=("Line Item Value", "sum"),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
)
mode_summary = mode_summary.rename(
    columns={"Shipment Mode": "modo_de_envío"}
).sort_values("cumplimiento_a_tiempo")
mode_summary.to_csv(SUBMISSION_DIR / "mode_summary.csv", index=False)
mode_summary

In [ ]:
# ¿Qué modos de envío requieren revisar primero su cumplimiento de la promesa de entrega?

px.bar(
    mode_summary,
    x="cumplimiento_a_tiempo",
    y="modo_de_envío",
    orientation="h",
    text="envíos",
    title="Cumplimiento a tiempo por modo de envío",
    labels={
        "cumplimiento_a_tiempo": "Proporción a tiempo",
        "modo_de_envío": "Modo de envío",
        "envíos": "Envíos",
    },
    color_discrete_sequence=["#f28e2b"],
).update_xaxes(tickformat=".0%")

In [ ]:
# ¿Qué países deben compararse con una base mínima de envíos para evitar conclusiones por pocos casos?

minimum_shipments = 50
country_summary = (
    shipments.groupby("Country")
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        demora_promedio_días=("DeliveryDaysVsSchedule", "mean"),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
)
country_summary = country_summary.query("envíos >= @minimum_shipments").sort_values(
    "cumplimiento_a_tiempo"
)
country_summary.to_csv(SUBMISSION_DIR / "country_summary.csv", index=False)
country_summary.head(10)

In [ ]:
# ¿En qué países se concentra el mayor valor de envíos tardíos entre los segmentos comparables?

priority_countries = country_summary.nlargest(
    10, "valor_enviado_tarde_usd"
).sort_values("valor_enviado_tarde_usd")
priority_countries.to_csv(SUBMISSION_DIR / "priority_countries.csv", index=False)
px.bar(
    priority_countries,
    x="valor_enviado_tarde_usd",
    y="Country",
    orientation="h",
    text="envíos",
    title="Valor expuesto a demora: diez países prioritarios",
    labels={
        "valor_enviado_tarde_usd": "Valor enviado tarde (USD)",
        "Country": "País",
        "envíos": "Envíos",
    },
    color_discrete_sequence=["#e15759"],
)

In [ ]:
# ¿Qué combinaciones país–modo revelan patrones de incumplimiento que un promedio global oculta?

country_mode = (
    shipments.groupby(["Country", "Shipment Mode"], dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
    )
    .reset_index()
    .query("envíos >= 20")
)
country_mode.to_csv(SUBMISSION_DIR / "country_mode_summary.csv", index=False)
country_mode_heatmap = country_mode.pivot(
    index="Country", columns="Shipment Mode", values="cumplimiento_a_tiempo"
)
px.imshow(
    country_mode_heatmap,
    aspect="auto",
    color_continuous_scale="Blues",
    zmin=0,
    zmax=1,
    title="Cumplimiento a tiempo por país y modo de envío (mínimo 20 envíos)",
    labels={"x": "Modo de envío", "y": "País", "color": "Cumplimiento"},
).update_coloraxes(colorbar_tickformat=".0%")

In [ ]:
# ¿Cómo evoluciona el cumplimiento mensual sin confundir un cambio de mezcla con una mejora operativa?

monthly_summary = (
    shipments.groupby("DeliveryMonth")
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
    )
    .reset_index()
)
monthly_summary.to_csv(SUBMISSION_DIR / "monthly_summary.csv", index=False)
px.line(
    monthly_summary,
    x="DeliveryMonth",
    y="cumplimiento_a_tiempo",
    markers=True,
    title="Cumplimiento mensual de la fecha prometida",
    labels={
        "DeliveryMonth": "Mes de entrega",
        "cumplimiento_a_tiempo": "Proporción a tiempo",
    },
    color_discrete_sequence=["#4e79a7"],
).update_yaxes(tickformat=".0%")

In [ ]:
# ¿Qué segmentos país–modo se deben priorizar para reducir riesgo, considerando volumen, cumplimiento y valor tardío?

priority_segments = (
    shipments.groupby(["Country", "Shipment Mode"], dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
    .query("envíos >= 30")
)
priority_segments.to_csv(SUBMISSION_DIR / "priority_segments.csv", index=False)
priority_segments.sort_values(
    ["valor_enviado_tarde_usd", "cumplimiento_a_tiempo"], ascending=[False, True]
).head(10)

In [ ]:
# ¿Qué proporción del valor de cada modo tiene flete numérico disponible para analizar eficiencia de costo?

freight_by_mode = (
    shipments.groupby("Shipment Mode", dropna=False)
    .agg(
        envíos=("ID", "size"),
        valor_total_usd=("Line Item Value", "sum"),
        valor_con_flete_numérico_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "HasNumericFreightCost"]].sum(),
        ),
        flete_total_usd=("FreightCostUSD", "sum"),
    )
    .reset_index()
)
freight_by_mode["cobertura_flete"] = (
    freight_by_mode["valor_con_flete_numérico_usd"] / freight_by_mode["valor_total_usd"]
)
freight_by_mode.to_csv(SUBMISSION_DIR / "freight_by_mode.csv", index=False)
freight_by_mode